# Assignment 03 — Bài toán 1: Chẩn đoán Đái tháo đường bằng Mạng nơ-ron sâu 5 tầng

**Môn học:** Intelligent System Development — TS. Trần Đình Quế
**Sinh viên:** Đinh Hải Triều — B23DCCN843 — Lớp 06

---

## Mục tiêu

1. Cài đặt **mạng nơ-ron sâu 5 tầng** hoàn toàn bằng **NumPy from scratch**
   (forward, backward, Adam) và một bản **PyTorch** tương đương để đối chiếu.
2. Đối sánh Deep Learning với các mô hình **Học máy truyền thống** (Logistic
   Regression, Decision Tree, Random Forest) trên cùng một tập kiểm thử.
3. Trực quan hoá đường cong huấn luyện, ma trận nhầm lẫn, ROC.
4. **Xuất trọng số ra `model_deep.json`** và nhúng vào web app để chạy
   serverless trên Vercel (suy luận 100% bằng JavaScript phía trình duyệt).

**Bài toán:** Binary Classification — dự đoán `Outcome` ∈ {0, 1}
(0 = không tiểu đường, 1 = tiểu đường) từ 8 chỉ số lâm sàng.

In [1]:
import json
import os
import time
import pathlib

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
)

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 150,
    "font.family": "DejaVu Sans",
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
sns.set_palette("deep")

ROOT = pathlib.Path.cwd()
FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)
print("Thư mục làm việc :", ROOT)
print("Thư mục hình vẽ  :", FIG)

Thư mục làm việc : C:\Users\admin\Downloads\bt-thay quế\tuan 1\website_chandoan_tieu_duong
Thư mục hình vẽ  : C:\Users\admin\Downloads\bt-thay quế\tuan 1\website_chandoan_tieu_duong\figures


## 1. Nạp dữ liệu và khảo sát ban đầu (EDA)

Bộ dữ liệu **Pima Indians Diabetes**: 768 bệnh nhân nữ gốc Pima, 8 đặc trưng
lâm sàng, nhãn nhị phân `Outcome`.

In [2]:
df = pd.read_csv(ROOT / "data" / "pima_diabetes.csv")
print("Kích thước:", df.shape)
print("\nPhân bố nhãn:")
print(df["Outcome"].value_counts().rename({0: "Không tiểu đường", 1: "Tiểu đường"}))
print("\nTỉ lệ dương tính: %.2f%%" % (100 * df["Outcome"].mean()))
df.head()

Kích thước: (768, 9)

Phân bố nhãn:
Outcome
Không tiểu đường    500
Tiểu đường          268
Name: count, dtype: int64

Tỉ lệ dương tính: 34.90%


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
df.describe().T[["mean", "std", "min", "50%", "max"]].round(2)

,mean,std,min,50%,max
Pregnancies,3.85,3.37,0.00,3.00,17.00
Glucose,120.89,31.97,0.00,117.00,199.00
BloodPressure,69.11,19.36,0.00,72.00,122.00
SkinThickness,20.54,15.95,0.00,23.00,99.00
Insulin,79.80,115.24,0.00,30.50,846.00
BMI,31.99,7.88,0.00,32.00,67.10
DiabetesPedigreeFunction,0.47,0.33,0.08,0.37,2.42
Age,33.24,11.76,21.00,29.00,81.00
Outcome,0.35,0.48,0.00,0.00,1.00


### 1.1. Phát hiện giá trị thiếu bị mã hoá thành 0

Trong bộ Pima, giá trị `0` ở các cột sinh lý là **bất khả thi về mặt y học**
(không ai có Glucose = 0 hay BMI = 0). Đây thực chất là **missing value bị mã
hoá nhầm** — nếu để nguyên, mạng nơ-ron sẽ học phải các mẫu giả.

In [4]:
ZERO_AS_NAN = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
miss = pd.DataFrame({
    "Số giá trị 0": [(df[c] == 0).sum() for c in ZERO_AS_NAN],
    "Tỉ lệ (%)": [round(100 * (df[c] == 0).mean(), 2) for c in ZERO_AS_NAN],
}, index=ZERO_AS_NAN)
miss

,Số giá trị 0,Tỉ lệ (%)
Glucose,5,0.65
BloodPressure,35,4.56
SkinThickness,227,29.56
Insulin,374,48.70
BMI,11,1.43


In [5]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# (a) Phân bố nhãn
counts = df["Outcome"].value_counts().sort_index()
axes[0, 0].bar(["Không tiểu đường (0)", "Tiểu đường (1)"], counts.values,
               color=["#3b82f6", "#ef4444"], width=0.55)
for i, v in enumerate(counts.values):
    axes[0, 0].text(i, v + 8, f"{v}\n({100*v/len(df):.1f}%)", ha="center", fontweight="bold")
axes[0, 0].set_title("(a) Phân bố nhãn — dữ liệu mất cân bằng nhẹ 65/35", fontweight="bold")
axes[0, 0].set_ylabel("Số bệnh nhân")
axes[0, 0].set_ylim(0, 620)

# (b) Glucose theo nhãn
for lbl, color, name in [(0, "#3b82f6", "Không tiểu đường"), (1, "#ef4444", "Tiểu đường")]:
    sub = df.loc[df["Outcome"] == lbl, "Glucose"].replace(0, np.nan).dropna()
    axes[0, 1].hist(sub, bins=28, alpha=0.6, color=color, label=name)
axes[0, 1].set_title("(b) Glucose — đặc trưng phân tách mạnh nhất", fontweight="bold")
axes[0, 1].set_xlabel("Glucose (mg/dL)"); axes[0, 1].set_ylabel("Tần suất"); axes[0, 1].legend()

# (c) Tỉ lệ giá trị 0
axes[1, 0].barh(miss.index, miss["Tỉ lệ (%)"], color="#f59e0b")
for i, v in enumerate(miss["Tỉ lệ (%)"]):
    axes[1, 0].text(v + 0.6, i, f"{v}%", va="center", fontweight="bold")
axes[1, 0].set_title("(c) Tỉ lệ giá trị 0 bất khả thi (missing trá hình)", fontweight="bold")
axes[1, 0].set_xlabel("% số mẫu"); axes[1, 0].set_xlim(0, 55)

# (d) Ma trận tương quan
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            ax=axes[1, 1], cbar_kws={"shrink": 0.8}, annot_kws={"size": 7})
axes[1, 1].set_title("(d) Ma trận tương quan Pearson", fontweight="bold")
axes[1, 1].grid(False)

plt.tight_layout()
plt.savefig(FIG / "p1_fig1_eda.png", bbox_inches="tight")
plt.show()

C:\Users\admin\AppData\Local\Temp\ipykernel_23704\270685834.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Tiền xử lý — chống rò rỉ dữ liệu (Data Leakage)

Quy tắc bắt buộc: **mọi thống kê (median, mean, std) chỉ được ước lượng trên
tập huấn luyện**, rồi áp dụng nguyên si sang tập validation và test.

In [6]:
X_raw = df.drop(columns=["Outcome"]).copy()
y = df["Outcome"].values.astype(float).reshape(-1, 1)
FEATURES = list(X_raw.columns)

# 0 -> NaN cho các cột sinh lý
for c in ZERO_AS_NAN:
    X_raw[c] = X_raw[c].replace(0, np.nan)

# Chia 70% train / 15% val / 15% test, stratify theo nhãn
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X_raw, y, test_size=0.15, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.1765, random_state=SEED, stratify=y_tmp)

# Median CHỈ tính trên train
medians = X_train.median()
X_train = X_train.fillna(medians)
X_val = X_val.fillna(medians)
X_test = X_test.fillna(medians)

scaler = StandardScaler().fit(X_train)
Xtr = scaler.transform(X_train)
Xva = scaler.transform(X_val)
Xte = scaler.transform(X_test)

print(f"Train: {Xtr.shape}  |  Val: {Xva.shape}  |  Test: {Xte.shape}")
print(f"Tỉ lệ dương tính  train={y_train.mean():.3f}  val={y_val.mean():.3f}  test={y_test.mean():.3f}")
print("\nMedian ước lượng trên TRAIN:")
print(medians.round(2).to_string())

Train: (536, 8)  |  Val: (116, 8)  |  Test: (116, 8)
Tỉ lệ dương tính  train=0.349  val=0.353  test=0.345

Median ước lượng trên TRAIN:
Pregnancies                   3.00
Glucose                     118.00
BloodPressure                72.00
SkinThickness                28.00
Insulin                     125.50
BMI                          32.00
DiabetesPedigreeFunction      0.38
Age                          29.00


## 3. Cài đặt mạng nơ-ron sâu 5 tầng bằng NumPy (from scratch)

### Kiến trúc

```
Input(8) → [W1] 128 → ReLU → [W2] 64 → ReLU → [W3] 32 → ReLU
         → [W4] 16 → ReLU → [W5] 1 → Sigmoid
```

### Bốn bước của một vòng lặp huấn luyện

| Bước | Tên | Công thức |
|---|---|---|
| 1 | Forward  | $A^{[l]} = \text{ReLU}(A^{[l-1]}W^{[l]} + b^{[l]})$ |
| 2 | Loss     | $\mathcal{L} = -\frac{1}{n}\sum [y\log\hat y + (1-y)\log(1-\hat y)]$ |
| 3 | Backward | $\frac{\partial\mathcal{L}}{\partial Z^{[5]}} = \hat y - y$, rồi lan truyền ngược |
| 4 | Update   | Adam: $\theta \leftarrow \theta - \eta\,\hat m/(\sqrt{\hat v}+\epsilon)$ |

In [7]:
from __future__ import annotations

import numpy as np

# ----------------------------------------------------------------------------
# 1. Hàm kích hoạt và đạo hàm
# ----------------------------------------------------------------------------


def relu(z):
    """f(z) = max(0, z) — phá vỡ tính tuyến tính, giữ gradient không bão hoà ở nhánh dương."""
    return np.maximum(0.0, z)


def relu_grad(z):
    """f'(z) = 1 nếu z > 0, ngược lại 0."""
    return (z > 0).astype(z.dtype)


def sigmoid(z):
    """Ổn định số học: tách nhánh z >= 0 và z < 0 để tránh exp() tràn số."""
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out


def softmax(z):
    """Trừ max theo hàng trước khi exp — kỹ thuật log-sum-exp chống tràn số."""
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


# ----------------------------------------------------------------------------
# 2. Mạng nơ-ron sâu 5 tầng
# ----------------------------------------------------------------------------


class DeepMLP:
    """5-Layer MLP thuần NumPy với Adam, mini-batch, He-init, L2 và Dropout."""

    def __init__(
        self,
        input_dim: int,
        hidden=(128, 64, 32, 16),
        output_dim: int = 1,
        task: str = "binary",
        lr: float = 1e-3,
        l2: float = 1e-4,
        dropout: float = 0.0,
        seed: int = 42,
        class_weight=None,
    ):
        assert task in {"binary", "regression", "multiclass"}
        self.task = task
        # class_weight: vector trọng số theo lớp, dùng để chống mất cân bằng dữ liệu.
        # Mỗi mẫu được nhân thêm w[y] trong cả hàm mất mát lẫn gradient, tương đương
        # "nhân bản" mẫu của lớp hiếm mà không phải sao chép dữ liệu thật.
        self.class_weight = None if class_weight is None else np.asarray(class_weight, dtype=float)
        self.lr = lr
        self.l2 = l2
        self.dropout = dropout
        self.dims = [input_dim, *hidden, output_dim]
        self.rng = np.random.default_rng(seed)

        # --- Khởi tạo He (Kaiming): Var(W) = 2/fan_in, phù hợp với ReLU ---
        self.W, self.b = [], []
        for i in range(len(self.dims) - 1):
            fan_in, fan_out = self.dims[i], self.dims[i + 1]
            self.W.append(self.rng.normal(0.0, np.sqrt(2.0 / fan_in), (fan_in, fan_out)))
            self.b.append(np.zeros(fan_out))

        # --- Trạng thái Adam (moment bậc 1 và bậc 2) ---
        self.mW = [np.zeros_like(w) for w in self.W]
        self.vW = [np.zeros_like(w) for w in self.W]
        self.mb = [np.zeros_like(b) for b in self.b]
        self.vb = [np.zeros_like(b) for b in self.b]
        self.t = 0

        self.history = {"train_loss": [], "val_loss": [], "train_metric": [], "val_metric": []}

    # ----------------------------- forward ---------------------------------
    def forward(self, X, training: bool = False):
        """Trả về (output, cache). cache giữ A/Z của từng tầng để lan truyền ngược."""
        A = X
        As, Zs, masks = [A], [], []
        n_layers = len(self.W)

        for i in range(n_layers - 1):  # 4 tầng ẩn
            Z = A @ self.W[i] + self.b[i]
            A = relu(Z)
            if training and self.dropout > 0.0:
                # Inverted dropout: chia cho keep_prob ngay lúc train nên lúc
                # suy luận không cần chỉnh gì — trọng số xuất ra JSON dùng trực tiếp.
                keep = 1.0 - self.dropout
                mask = (self.rng.random(A.shape) < keep) / keep
                A = A * mask
                masks.append(mask)
            else:
                masks.append(None)
            Zs.append(Z)
            As.append(A)

        # Tầng 5 — output head
        Zout = A @ self.W[-1] + self.b[-1]
        Zs.append(Zout)
        if self.task == "binary":
            out = sigmoid(Zout)
        elif self.task == "multiclass":
            out = softmax(Zout)
        else:
            out = Zout
        As.append(out)
        return out, (As, Zs, masks)

    # ------------------------------ loss -----------------------------------
    def _sample_w(self, y_true):
        """Trọng số của từng mẫu suy ra từ class_weight (shape (n, 1))."""
        if self.class_weight is None or self.task != "multiclass":
            return None
        return self.class_weight[y_true.argmax(1)].reshape(-1, 1)

    def loss(self, y_pred, y_true):
        n = y_true.shape[0]
        if self.task == "binary":
            eps = 1e-12
            base = -np.mean(
                y_true * np.log(y_pred + eps) + (1 - y_true) * np.log(1 - y_pred + eps)
            )
        elif self.task == "multiclass":
            eps = 1e-12
            per_sample = -np.sum(y_true * np.log(y_pred + eps), axis=1, keepdims=True)
            w = self._sample_w(y_true)
            base = float(np.sum(per_sample if w is None else per_sample * w) / n)
        else:
            base = np.mean((y_pred - y_true) ** 2)
        reg = self.l2 * sum(np.sum(w * w) for w in self.W) / (2 * n)
        return base + reg

    # ---------------------------- backward ---------------------------------
    def backward(self, cache, y_true):
        As, Zs, masks = cache
        n = y_true.shape[0]
        n_layers = len(self.W)

        # Với cả 3 head, đạo hàm của loss theo pre-activation cuối cùng rút gọn
        # về (y_hat - y). Đây là lý do ta ghép Sigmoid/Softmax với Cross-Entropy
        # và Linear với MSE.
        dZ = (As[-1] - y_true) / n
        if self.task == "regression":
            dZ = 2.0 * dZ
        w = self._sample_w(y_true)
        if w is not None:
            dZ = dZ * w

        dW = [None] * n_layers
        db = [None] * n_layers

        for i in range(n_layers - 1, -1, -1):
            dW[i] = As[i].T @ dZ + self.l2 * self.W[i] / n
            db[i] = dZ.sum(axis=0)
            if i > 0:
                dA = dZ @ self.W[i].T
                if masks[i - 1] is not None:
                    dA = dA * masks[i - 1]
                dZ = dA * relu_grad(Zs[i - 1])
        return dW, db

    # ------------------------------ Adam -----------------------------------
    def _adam(self, dW, db, beta1=0.9, beta2=0.999, eps=1e-8):
        self.t += 1
        for i in range(len(self.W)):
            self.mW[i] = beta1 * self.mW[i] + (1 - beta1) * dW[i]
            self.vW[i] = beta2 * self.vW[i] + (1 - beta2) * (dW[i] ** 2)
            mhat = self.mW[i] / (1 - beta1**self.t)
            vhat = self.vW[i] / (1 - beta2**self.t)
            self.W[i] -= self.lr * mhat / (np.sqrt(vhat) + eps)

            self.mb[i] = beta1 * self.mb[i] + (1 - beta1) * db[i]
            self.vb[i] = beta2 * self.vb[i] + (1 - beta2) * (db[i] ** 2)
            mhat = self.mb[i] / (1 - beta1**self.t)
            vhat = self.vb[i] / (1 - beta2**self.t)
            self.b[i] -= self.lr * mhat / (np.sqrt(vhat) + eps)

    # ------------------------------ metric ---------------------------------
    def _metric(self, X, y):
        p, _ = self.forward(X, training=False)
        if self.task == "binary":
            return float(np.mean((p >= 0.5).astype(int) == y))
        if self.task == "multiclass":
            return float(np.mean(p.argmax(1) == y.argmax(1)))
        ss_res = np.sum((y - p) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        return float(1 - ss_res / ss_tot)  # R^2

    # ------------------------------- fit -----------------------------------
    def fit(self, X, y, X_val=None, y_val=None, epochs=200, batch_size=32,
            verbose_every=20, patience=None):
        """Chu trình huấn luyện 4 bước: Forward -> Loss -> Backward -> Update."""
        n = X.shape[0]
        best_val, best_state, wait = np.inf, None, 0

        for ep in range(1, epochs + 1):
            idx = self.rng.permutation(n)
            Xs, ys = X[idx], y[idx]

            for s in range(0, n, batch_size):
                xb, yb = Xs[s:s + batch_size], ys[s:s + batch_size]
                _, cache = self.forward(xb, training=True)          # (1) Forward
                dW, db = self.backward(cache, yb)                   # (3) Backward
                self._adam(dW, db)                                  # (4) Update

            tr_pred, _ = self.forward(X, training=False)
            tr_loss = self.loss(tr_pred, y)                         # (2) Loss
            self.history["train_loss"].append(tr_loss)
            self.history["train_metric"].append(self._metric(X, y))

            if X_val is not None:
                va_pred, _ = self.forward(X_val, training=False)
                va_loss = self.loss(va_pred, y_val)
                self.history["val_loss"].append(va_loss)
                self.history["val_metric"].append(self._metric(X_val, y_val))

                if patience is not None:
                    if va_loss < best_val - 1e-6:
                        best_val, wait = va_loss, 0
                        best_state = ([w.copy() for w in self.W], [b.copy() for b in self.b])
                    else:
                        wait += 1
                        if wait >= patience:
                            if verbose_every:
                                print(f"  ⏹ Early stopping tại epoch {ep} (val_loss tốt nhất = {best_val:.4f})")
                            break

            if verbose_every and (ep % verbose_every == 0 or ep == 1):
                msg = f"  epoch {ep:4d} | train_loss={tr_loss:.4f} | train_metric={self.history['train_metric'][-1]:.4f}"
                if X_val is not None:
                    msg += f" | val_loss={self.history['val_loss'][-1]:.4f} | val_metric={self.history['val_metric'][-1]:.4f}"
                print(msg)

        if best_state is not None:
            self.W, self.b = best_state
        return self

    # ---------------------------- inference --------------------------------
    def predict_proba(self, X):
        out, _ = self.forward(X, training=False)
        return out

    def predict(self, X):
        out = self.predict_proba(X)
        if self.task == "binary":
            return (out >= 0.5).astype(int)
        if self.task == "multiclass":
            return out.argmax(1)
        return out

    # -------------------------- xuất ra JSON -------------------------------
    def n_params(self):
        return sum(w.size for w in self.W) + sum(b.size for b in self.b)

    def to_dict(self, decimals: int = 6):
        """Đóng gói trọng số về dict thuần Python để ghi ra model.json cho web."""
        return {
            "architecture": self.dims,
            "task": self.task,
            "activation": "relu",
            "n_params": int(self.n_params()),
            "layers": [
                {"W": np.round(w, decimals).tolist(), "b": np.round(bb, decimals).tolist()}
                for w, bb in zip(self.W, self.b)
            ],
        }


# ----------------------------------------------------------------------------
# 3. Tiện ích dùng chung
# ----------------------------------------------------------------------------


def one_hot(y, n_classes):
    out = np.zeros((len(y), n_classes))
    out[np.arange(len(y)), y] = 1.0
    return out


def forward_reference(model_dict, x):
    """Bản tham chiếu của thuật toán suy luận sẽ viết lại bằng JavaScript trên web.

    Dùng trong notebook để kiểm tra parity: NumPy <-> JSON <-> JavaScript.
    """
    a = np.asarray(x, dtype=float)
    layers = model_dict["layers"]
    for i, layer in enumerate(layers):
        z = a @ np.array(layer["W"]) + np.array(layer["b"])
        if i < len(layers) - 1:
            a = np.maximum(0.0, z)
        else:
            a = z
    if model_dict["task"] == "binary":
        return sigmoid(a)
    if model_dict["task"] == "multiclass":
        return softmax(a.reshape(1, -1))[0]
    return a

### 3.1. Bảng đặc tả Tensor Shapes và Không gian tham số

In [8]:
HP = dict(lr=1e-3, l2=5e-2, dropout=0.40, batch_size=32, epochs=400, patience=60)
print("Siêu tham số được chọn bằng grid-search trên tập VALIDATION (tiêu chí ROC-AUC):")
print(HP)

mlp = DeepMLP(input_dim=Xtr.shape[1], hidden=(128, 64, 32, 16), output_dim=1,
              task="binary", lr=HP["lr"], l2=HP["l2"], dropout=HP["dropout"], seed=SEED)

rows = []
names = ["Layer 1 (Input→H1)", "Layer 2 (H1→H2)", "Layer 3 (H2→H3)",
         "Layer 4 (H3→H4)", "Layer 5 (H4→Output)"]
acts = ["ReLU", "ReLU", "ReLU", "ReLU", "Sigmoid"]
for i, (nm, act) in enumerate(zip(names, acts)):
    fi, fo = mlp.dims[i], mlp.dims[i + 1]
    rows.append({
        "Tầng": nm,
        "W shape": f"({fi}, {fo})",
        "b shape": f"({fo},)",
        "Output shape": f"(batch, {fo})",
        "Kích hoạt": act,
        "Số tham số": fi * fo + fo,
    })
shape_table = pd.DataFrame(rows)
shape_table.loc[len(shape_table)] = ["TỔNG", "", "", "", "", shape_table["Số tham số"].sum()]
shape_table

Siêu tham số được chọn bằng grid-search trên tập VALIDATION (tiêu chí ROC-AUC):
{'lr': 0.001, 'l2': 0.05, 'dropout': 0.4, 'batch_size': 32, 'epochs': 400, 'patience': 60}


,Tầng,W shape,b shape,Output shape,Kích hoạt,Số tham số
0,Layer 1 (Input→H1),"(8, 128)","(128,)","(batch, 128)",ReLU,1152
1,Layer 2 (H1→H2),"(128, 64)","(64,)","(batch, 64)",ReLU,8256
2,Layer 3 (H2→H3),"(64, 32)","(32,)","(batch, 32)",ReLU,2080
3,Layer 4 (H3→H4),"(32, 16)","(16,)","(batch, 16)",ReLU,528
4,Layer 5 (H4→Output),"(16, 1)","(1,)","(batch, 1)",Sigmoid,17
5,TỔNG,,,,,12033


## 4. Huấn luyện mô hình NumPy from scratch

In [9]:
t0 = time.perf_counter()
mlp.fit(Xtr, y_train, Xva, y_val, epochs=HP["epochs"], batch_size=HP["batch_size"],
        verbose_every=50, patience=HP["patience"])
numpy_time = time.perf_counter() - t0
print(f"\n⏱ Thời gian huấn luyện NumPy: {numpy_time:.2f}s | Tham số: {mlp.n_params():,}")

  epoch    1 | train_loss=0.6431 | train_metric=0.6045 | val_loss=0.7271 | val_metric=0.6207


  epoch   50 | train_loss=0.4765 | train_metric=0.7854 | val_loss=0.5774 | val_metric=0.7500


  epoch  100 | train_loss=0.4195 | train_metric=0.8060 | val_loss=0.5448 | val_metric=0.7759


  epoch  150 | train_loss=0.3769 | train_metric=0.8190 | val_loss=0.5387 | val_metric=0.7931


  ⏹ Early stopping tại epoch 194 (val_loss tốt nhất = 0.5292)

⏱ Thời gian huấn luyện NumPy: 3.76s | Tham số: 12,033


## 5. Bản cài đặt PyTorch tương đương (đối chiếu)

Cùng kiến trúc, cùng optimizer, cùng seed — điểm khác biệt duy nhất là
**backward được autograd sinh tự động** thay vì viết tay.

In [10]:
import torch
import torch.nn as nn

torch.manual_seed(SEED)

D = HP["dropout"]
torch_net = nn.Sequential(
    nn.Linear(Xtr.shape[1], 128), nn.ReLU(), nn.Dropout(D),
    nn.Linear(128, 64), nn.ReLU(), nn.Dropout(D),
    nn.Linear(64, 32), nn.ReLU(), nn.Dropout(D),
    nn.Linear(32, 16), nn.ReLU(), nn.Dropout(D),
    nn.Linear(16, 1),
)
print(torch_net)
print("Tổng tham số PyTorch:", sum(p.numel() for p in torch_net.parameters()))

Sequential(
  (0): Linear(in_features=8, out_features=128, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.4, inplace=False)
  (3): Linear(in_features=128, out_features=64, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.4, inplace=False)
  (6): Linear(in_features=64, out_features=32, bias=True)
  (7): ReLU()
  (8): Dropout(p=0.4, inplace=False)
  (9): Linear(in_features=32, out_features=16, bias=True)
  (10): ReLU()
  (11): Dropout(p=0.4, inplace=False)
  (12): Linear(in_features=16, out_features=1, bias=True)
)
Tổng tham số PyTorch: 12033


In [11]:
# ⚠ Điểm khác biệt quan trọng giữa hai bản cài đặt:
# Trong bản NumPy, số hạng L2 được cộng vào gradient dưới dạng (l2 * W / n) với n là
# kích thước mini-batch — tức là đã "trung bình hoá" theo batch giống như hàm loss.
# torch.optim.Adam thì cộng thẳng (weight_decay * W) vào gradient, KHÔNG chia cho n.
# Muốn hai bản tương đương thì phải quy đổi weight_decay = l2 / batch_size.
# (Nếu truyền thẳng weight_decay=0.05, hình phạt mạnh gấp 32 lần và mạng sẽ sụp về
#  hằng số — dự đoán mọi bệnh nhân đều âm tính, AUC = 0.5.)
WEIGHT_DECAY = HP["l2"] / HP["batch_size"]
print(f"l2 (NumPy) = {HP['l2']}  ->  weight_decay (PyTorch) = {WEIGHT_DECAY:.6f}")

opt = torch.optim.Adam(torch_net.parameters(), lr=HP["lr"], weight_decay=WEIGHT_DECAY)
lossfn = nn.BCEWithLogitsLoss()

Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
ytr_t = torch.tensor(y_train, dtype=torch.float32)
Xva_t = torch.tensor(Xva, dtype=torch.float32)
yva_t = torch.tensor(y_val, dtype=torch.float32)
Xte_t = torch.tensor(Xte, dtype=torch.float32)

torch_hist = {"train_loss": [], "val_loss": []}
t0 = time.perf_counter()
n = len(Xtr_t)
best_state, best_val, wait = None, np.inf, 0
for ep in range(1, HP["epochs"] + 1):
    torch_net.train()
    perm = torch.randperm(n)
    for s in range(0, n, HP["batch_size"]):
        idx = perm[s:s + HP["batch_size"]]
        opt.zero_grad()
        loss = lossfn(torch_net(Xtr_t[idx]), ytr_t[idx])
        loss.backward()
        opt.step()
    torch_net.eval()
    with torch.no_grad():
        tl = lossfn(torch_net(Xtr_t), ytr_t).item()
        vl = lossfn(torch_net(Xva_t), yva_t).item()
    torch_hist["train_loss"].append(tl)
    torch_hist["val_loss"].append(vl)
    if vl < best_val - 1e-6:
        best_val, wait = vl, 0
        best_state = {k: v.clone() for k, v in torch_net.state_dict().items()}
    else:
        wait += 1
        if wait >= HP["patience"]:
            print(f"  ⏹ Early stopping tại epoch {ep}")
            break
    if ep % 50 == 0:
        print(f"  epoch {ep:4d} | train_loss={tl:.4f} | val_loss={vl:.4f}")
torch_net.load_state_dict(best_state)
torch_time = time.perf_counter() - t0
print(f"\n⏱ Thời gian huấn luyện PyTorch: {torch_time:.2f}s")

l2 (NumPy) = 0.05  ->  weight_decay (PyTorch) = 0.001563


  epoch   50 | train_loss=0.3630 | val_loss=0.4976


  ⏹ Early stopping tại epoch 79

⏱ Thời gian huấn luyện PyTorch: 4.15s


## 6. Các mô hình Học máy truyền thống (baseline đối sánh)

In [12]:
classical = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=50, max_depth=6, random_state=SEED),
}
for name, m in classical.items():
    m.fit(Xtr, y_train.ravel())
    print(f"{name:22s} val acc = {m.score(Xva, y_val.ravel()):.4f}")

Logistic Regression    val acc = 0.7759
Decision Tree          val acc = 0.7241
Random Forest          val acc = 0.7672


## 7. Đánh giá trên tập kiểm thử (Test Set)

In [13]:
def evaluate(name, y_true, proba, thr=0.5):
    pred = (proba >= thr).astype(int)
    return {
        "Mô hình": name,
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, proba),
    }

yte = y_test.ravel()
proba_np = mlp.predict_proba(Xte).ravel()
with torch.no_grad():
    proba_torch = torch.sigmoid(torch_net(Xte_t)).numpy().ravel()

results, probas = [], {}
for name, m in classical.items():
    p = m.predict_proba(Xte)[:, 1]
    probas[name] = p
    results.append(evaluate(name, yte, p))

probas["Deep MLP-5 (NumPy)"] = proba_np
probas["Deep MLP-5 (PyTorch)"] = proba_torch
results.append(evaluate("Deep MLP-5 (NumPy)", yte, proba_np))
results.append(evaluate("Deep MLP-5 (PyTorch)", yte, proba_torch))

res_df = pd.DataFrame(results).set_index("Mô hình").round(4)
res_df.sort_values("F1", ascending=False)

,Accuracy,Precision,Recall,F1,ROC-AUC
Mô hình,,,,,
Random Forest,0.7672,0.6757,0.625,0.6494,0.8293
Deep MLP-5 (NumPy),0.7500,0.6341,0.650,0.6420,0.8382
Deep MLP-5 (PyTorch),0.7328,0.6098,0.625,0.6173,0.8418
Logistic Regression,0.7069,0.5789,0.550,0.5641,0.8345
Decision Tree,0.7155,0.6061,0.500,0.5479,0.7709


In [14]:
print("Báo cáo chi tiết — Deep MLP-5 (NumPy from scratch)\n")
print(classification_report(yte, (proba_np >= 0.5).astype(int),
                            target_names=["Không tiểu đường", "Tiểu đường"], digits=4))

Báo cáo chi tiết — Deep MLP-5 (NumPy from scratch)

                  precision    recall  f1-score   support

Không tiểu đường     0.8133    0.8026    0.8079        76
      Tiểu đường     0.6341    0.6500    0.6420        40

        accuracy                         0.7500       116
       macro avg     0.7237    0.7263    0.7250       116
    weighted avg     0.7515    0.7500    0.7507       116



### 7.1. Hiệu chỉnh ngưỡng quyết định (Threshold Tuning)

Ngưỡng mặc định 0.5 giả định hai loại sai lầm có chi phí ngang nhau. Trong sàng
lọc y khoa điều đó **sai**: bỏ sót một ca tiểu đường (False Negative) nguy hiểm
hơn nhiều so với việc gọi nhầm một người khoẻ đi xét nghiệm lại (False Positive).

Ta dò ngưỡng tối ưu theo **F1 trên tập validation** (không bao giờ chạm vào test),
rồi áp cùng thủ tục đó cho **tất cả** các mô hình để so sánh công bằng.

In [15]:
val_probas = {name: m.predict_proba(Xva)[:, 1] for name, m in classical.items()}
val_probas["Deep MLP-5 (NumPy)"] = mlp.predict_proba(Xva).ravel()
with torch.no_grad():
    val_probas["Deep MLP-5 (PyTorch)"] = torch.sigmoid(torch_net(Xva_t)).numpy().ravel()

grid = np.linspace(0.05, 0.95, 181)
best_thr = {}
for name, pv in val_probas.items():
    f1s = [f1_score(y_val.ravel(), (pv >= t).astype(int), zero_division=0) for t in grid]
    best_thr[name] = float(grid[int(np.argmax(f1s))])

tuned = [evaluate(name, yte, probas[name], thr=best_thr[name]) for name in probas]
tuned_df = pd.DataFrame(tuned).set_index("Mô hình").round(4)
tuned_df.insert(0, "Ngưỡng*", [best_thr[n] for n in tuned_df.index])
print("Ngưỡng * được chọn trên VALIDATION, đánh giá trên TEST:\n")
tuned_df.sort_values("F1", ascending=False)

Ngưỡng * được chọn trên VALIDATION, đánh giá trên TEST:



,Ngưỡng*,Accuracy,Precision,Recall,F1,ROC-AUC
Mô hình,,,,,,
Logistic Regression,0.240,0.7500,0.5902,0.900,0.7129,0.8345
Deep MLP-5 (PyTorch),0.430,0.7759,0.6522,0.750,0.6977,0.8418
Deep MLP-5 (NumPy),0.430,0.7586,0.6200,0.775,0.6889,0.8382
Decision Tree,0.105,0.7328,0.5818,0.800,0.6737,0.7709
Random Forest,0.375,0.7328,0.5882,0.750,0.6593,0.8293


In [16]:
fig, ax = plt.subplots(figsize=(8, 4.5))
pv = val_probas["Deep MLP-5 (NumPy)"]
f1s = [f1_score(y_val.ravel(), (pv >= t).astype(int), zero_division=0) for t in grid]
recs = [recall_score(y_val.ravel(), (pv >= t).astype(int), zero_division=0) for t in grid]
pres = [precision_score(y_val.ravel(), (pv >= t).astype(int), zero_division=0) for t in grid]
ax.plot(grid, f1s, lw=2.4, color="#ef4444", label="F1")
ax.plot(grid, recs, lw=1.8, color="#3b82f6", ls="--", label="Recall")
ax.plot(grid, pres, lw=1.8, color="#22c55e", ls="--", label="Precision")
t_star = best_thr["Deep MLP-5 (NumPy)"]
ax.axvline(t_star, color="#111827", ls=":", lw=1.5)
ax.axvline(0.5, color="#9ca3af", ls=":", lw=1.5)
ax.annotate(f"Ngưỡng tối ưu = {t_star:.3f}", xy=(t_star, max(f1s)),
            xytext=(t_star + 0.06, max(f1s) + 0.06),
            arrowprops=dict(arrowstyle="->"), fontsize=9, fontweight="bold")
ax.text(0.5, 0.03, "mặc định 0.5", rotation=90, fontsize=8, color="#6b7280")
ax.set_xlabel("Ngưỡng quyết định"); ax.set_ylabel("Giá trị (trên tập validation)")
ax.set_title("Đánh đổi Precision – Recall theo ngưỡng (Deep MLP-5)", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(FIG / "p1_fig5_threshold.png", bbox_inches="tight")
plt.show()

C:\Users\admin\AppData\Local\Temp\ipykernel_23704\1102559594.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Trực quan hoá

### 8.1. Đường cong huấn luyện (Learning Curves)

In [17]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

ep = range(1, len(mlp.history["train_loss"]) + 1)
axes[0].plot(ep, mlp.history["train_loss"], label="Train loss", color="#2563eb", lw=2)
axes[0].plot(ep, mlp.history["val_loss"], label="Validation loss", color="#f97316", lw=2)
best_ep = int(np.argmin(mlp.history["val_loss"])) + 1
axes[0].axvline(best_ep, ls="--", color="#16a34a", alpha=0.8)
axes[0].annotate(f"Val loss thấp nhất\nepoch {best_ep}",
                 xy=(best_ep, min(mlp.history["val_loss"])),
                 xytext=(best_ep + 12, min(mlp.history["val_loss"]) + 0.06),
                 arrowprops=dict(arrowstyle="->", color="#16a34a"), fontsize=8, color="#16a34a")
axes[0].set_title("(a) Binary Cross-Entropy — NumPy from scratch", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend()

axes[1].plot(ep, mlp.history["train_metric"], label="Train accuracy", color="#2563eb", lw=2)
axes[1].plot(ep, mlp.history["val_metric"], label="Validation accuracy", color="#f97316", lw=2)
axes[1].set_title("(b) Accuracy theo epoch", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy"); axes[1].legend()

ep2 = range(1, len(torch_hist["train_loss"]) + 1)
axes[2].plot(ep2, torch_hist["train_loss"], label="Train loss (PyTorch)", color="#7c3aed", lw=2)
axes[2].plot(ep2, torch_hist["val_loss"], label="Val loss (PyTorch)", color="#dc2626", lw=2)
axes[2].set_title("(c) Bản PyTorch — cùng kiến trúc", fontweight="bold")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Loss"); axes[2].legend()

plt.tight_layout()
plt.savefig(FIG / "p1_fig2_curves.png", bbox_inches="tight")
plt.show()

C:\Users\admin\AppData\Local\Temp\ipykernel_23704\3645961314.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 8.2. Ma trận nhầm lẫn (Confusion Matrix)

In [18]:
panels = [
    ("Logistic Regression", 0.5, "Logistic Regression @0.5"),
    ("Random Forest", 0.5, "Random Forest @0.5"),
    ("Deep MLP-5 (NumPy)", 0.5, "Deep MLP-5 @0.5"),
    ("Deep MLP-5 (NumPy)", best_thr["Deep MLP-5 (NumPy)"],
     f"Deep MLP-5 @{best_thr['Deep MLP-5 (NumPy)']:.2f} (tuned)"),
]
fig, axes = plt.subplots(1, 4, figsize=(19, 4.2))
for ax, (name, thr, title) in zip(axes, panels):
    pred = (probas[name] >= thr).astype(int)
    cm = confusion_matrix(yte, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                xticklabels=["Dự đoán 0", "Dự đoán 1"],
                yticklabels=["Thực tế 0", "Thực tế 1"], annot_kws={"size": 14})
    acc = accuracy_score(yte, pred)
    rec = recall_score(yte, pred, zero_division=0)
    ax.set_title(f"{title}\nAcc={acc:.3f} | Recall={rec:.3f} | FN={cm[1,0]}",
                 fontweight="bold", fontsize=9.5)
    ax.grid(False)
plt.tight_layout()
plt.savefig(FIG / "p1_fig3_confusion.png", bbox_inches="tight")
plt.show()

C:\Users\admin\AppData\Local\Temp\ipykernel_23704\4119855871.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 8.3. Đường cong ROC và biểu đồ đối sánh

In [19]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {"Logistic Regression": "#0ea5e9", "Decision Tree": "#a855f7",
          "Random Forest": "#22c55e", "Deep MLP-5 (NumPy)": "#ef4444",
          "Deep MLP-5 (PyTorch)": "#f59e0b"}
for name, p in probas.items():
    fpr, tpr, _ = roc_curve(yte, p)
    axes[0].plot(fpr, tpr, lw=2, color=colors[name],
                 label=f"{name} (AUC={roc_auc_score(yte, p):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Ngẫu nhiên (AUC=0.5)")
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("(a) ROC Curve trên tập Test", fontweight="bold")
axes[0].legend(fontsize=8, loc="lower right")

metrics = ["Accuracy", "F1", "ROC-AUC"]
order = list(probas.keys())
x = np.arange(len(order)); w = 0.26
for i, mname in enumerate(metrics):
    vals = [res_df.loc[o, mname] for o in order]
    bars = axes[1].bar(x + (i - 1) * w, vals, w, label=mname)
    for b, v in zip(bars, vals):
        axes[1].text(b.get_x() + b.get_width() / 2, v + 0.008, f"{v:.3f}",
                     ha="center", fontsize=7, rotation=90)
axes[1].set_xticks(x)
axes[1].set_xticklabels([o.replace(" (", "\n(") for o in order], fontsize=8)
axes[1].set_ylim(0, 1.05); axes[1].set_ylabel("Giá trị")
axes[1].set_title("(b) Đối sánh Classical ML vs Deep Learning", fontweight="bold")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIG / "p1_fig4_roc_compare.png", bbox_inches="tight")
plt.show()

C:\Users\admin\AppData\Local\Temp\ipykernel_23704\2698096638.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Xuất mô hình sang JSON và nhúng vào Web App

Toàn bộ suy luận của MLP chỉ là **phép nhân ma trận + ReLU + Sigmoid** — hoàn
toàn tái hiện được bằng JavaScript thuần. Ta xuất trọng số ra `model_deep.json`
rồi nhúng thẳng vào `index.html` để Vercel phục vụ site tĩnh, không cần backend.

In [20]:
best_val_acc = max(mlp.history["val_metric"])
deep_bundle = mlp.to_dict(decimals=6)
deep_bundle.update({
    "model_name": "Deep MLP 5 tầng (NumPy from scratch)",
    "feature_names": FEATURES,
    "class_names": ["Không tiểu đường", "Tiểu đường"],
    "impute_median": [float(medians[c]) if c in ZERO_AS_NAN else None for c in FEATURES],
    "zero_as_nan": ZERO_AS_NAN,
    "scaler": {
        "mean_": scaler.mean_.tolist(),
        "scale_": scaler.scale_.tolist(),
    },
    "threshold": best_thr["Deep MLP-5 (NumPy)"],
    "metrics": {
        "accuracy": float(res_df.loc["Deep MLP-5 (NumPy)", "Accuracy"]),
        "precision": float(res_df.loc["Deep MLP-5 (NumPy)", "Precision"]),
        "recall": float(res_df.loc["Deep MLP-5 (NumPy)", "Recall"]),
        "f1": float(res_df.loc["Deep MLP-5 (NumPy)", "F1"]),
        "roc_auc": float(res_df.loc["Deep MLP-5 (NumPy)", "ROC-AUC"]),
    },
    "metrics_tuned": {
        "accuracy": float(tuned_df.loc["Deep MLP-5 (NumPy)", "Accuracy"]),
        "precision": float(tuned_df.loc["Deep MLP-5 (NumPy)", "Precision"]),
        "recall": float(tuned_df.loc["Deep MLP-5 (NumPy)", "Recall"]),
        "f1": float(tuned_df.loc["Deep MLP-5 (NumPy)", "F1"]),
    },
    "baseline_metrics": {k: {m: float(res_df.loc[k, m]) for m in
                            ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]}
                         for k in classical},
    "training": {
        "epochs_run": len(mlp.history["train_loss"]),
        "batch_size": HP["batch_size"], "lr": HP["lr"],
        "l2": HP["l2"], "dropout": HP["dropout"],
        "numpy_seconds": round(numpy_time, 2),
        "pytorch_seconds": round(torch_time, 2),
    },
    "history": {
        "train_loss": [round(v, 5) for v in mlp.history["train_loss"]],
        "val_loss": [round(v, 5) for v in mlp.history["val_loss"]],
        "train_acc": [round(v, 5) for v in mlp.history["train_metric"]],
        "val_acc": [round(v, 5) for v in mlp.history["val_metric"]],
    },
})

out_path = ROOT / "model_deep.json"
out_path.write_text(json.dumps(deep_bundle, ensure_ascii=False, separators=(",", ":")),
                    encoding="utf-8")
print(f"✅ Đã ghi {out_path.name} — {out_path.stat().st_size/1024:.1f} KB")

✅ Đã ghi model_deep.json — 118.7 KB


### 9.1. Kiểm tra parity: NumPy ↔ JSON

Chạy lại thuật toán suy luận "kiểu JavaScript" (chỉ dùng list/vòng lặp cơ bản)
trên đúng bundle JSON đã ghi và so với đầu ra gốc của mô hình.

In [21]:
reloaded = json.loads(out_path.read_text(encoding="utf-8"))
max_err = 0.0
for i in range(len(Xte)):
    ref = forward_reference(reloaded, Xte[i])
    max_err = max(max_err, abs(float(ref.ravel()[0]) - float(proba_np[i])))
print(f"Sai số tuyệt đối lớn nhất giữa mô hình gốc và bundle JSON: {max_err:.3e}")
assert max_err < 1e-5, "Parity FAILED"
print("✅ Parity PASSED — web app sẽ cho kết quả trùng khớp với notebook.")

Sai số tuyệt đối lớn nhất giữa mô hình gốc và bundle JSON: 5.883e-07
✅ Parity PASSED — web app sẽ cho kết quả trùng khớp với notebook.


### 9.2. Nhúng mô hình vào `index.html`

Thay dòng `const DEEP_MODEL = ...;` trong `index.html` bằng bundle JSON vừa xuất.

In [22]:
import re

html_path = ROOT / "index.html"
html = html_path.read_text(encoding="utf-8")
payload = json.dumps(deep_bundle, ensure_ascii=False, separators=(",", ":"))
new_line = f"const DEEP_MODEL = {payload};"

if re.search(r"^const DEEP_MODEL = .*;$", html, flags=re.M):
    html = re.sub(r"^const DEEP_MODEL = .*;$", lambda _: new_line, html, count=1, flags=re.M)
    html_path.write_text(html, encoding="utf-8")
    print(f"✅ Đã nhúng DEEP_MODEL vào index.html ({len(payload)/1024:.1f} KB)")
else:
    print("⚠ Không tìm thấy chốt `const DEEP_MODEL = ...;` trong index.html — bỏ qua bước nhúng.")

✅ Đã nhúng DEEP_MODEL vào index.html (118.7 KB)

## 10. Kết luận Bài toán 1

Bảng tổng hợp cuối cùng trên tập Test:

In [23]:
summary = res_df.copy()
summary["Nhóm"] = ["Classical ML"] * len(classical) + ["Deep Learning"] * 2
summary = summary[["Nhóm", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]]
summary.sort_values("F1", ascending=False)

,Nhóm,Accuracy,Precision,Recall,F1,ROC-AUC
Mô hình,,,,,,
Random Forest,Classical ML,0.7672,0.6757,0.625,0.6494,0.8293
Deep MLP-5 (NumPy),Deep Learning,0.7500,0.6341,0.650,0.6420,0.8382
Deep MLP-5 (PyTorch),Deep Learning,0.7328,0.6098,0.625,0.6173,0.8418
Logistic Regression,Classical ML,0.7069,0.5789,0.550,0.5641,0.8345
Decision Tree,Classical ML,0.7155,0.6061,0.500,0.5479,0.7709


In [24]:
print(f"""
TÓM TẮT BÀI TOÁN 1 — DIABETES PREDICTION
{'='*62}
Kiến trúc      : 8 → 128 → 64 → 32 → 16 → 1 (Sigmoid)
Tổng tham số   : {mlp.n_params():,}
Epoch đã chạy  : {len(mlp.history['train_loss'])} (early stopping, patience={HP['patience']})
Thời gian train: NumPy {numpy_time:.2f}s | PyTorch {torch_time:.2f}s
Test Accuracy  : {res_df.loc['Deep MLP-5 (NumPy)', 'Accuracy']:.4f}  (@0.5)
Test F1        : {res_df.loc['Deep MLP-5 (NumPy)', 'F1']:.4f}  (@0.5)
Test ROC-AUC   : {res_df.loc['Deep MLP-5 (NumPy)', 'ROC-AUC']:.4f}
Ngưỡng tối ưu  : {best_thr['Deep MLP-5 (NumPy)']:.3f} -> Acc={tuned_df.loc['Deep MLP-5 (NumPy)','Accuracy']:.4f}, F1={tuned_df.loc['Deep MLP-5 (NumPy)','F1']:.4f}, Recall={tuned_df.loc['Deep MLP-5 (NumPy)','Recall']:.4f}
{'='*62}
""")


TÓM TẮT BÀI TOÁN 1 — DIABETES PREDICTION
Kiến trúc      : 8 → 128 → 64 → 32 → 16 → 1 (Sigmoid)
Tổng tham số   : 12,033
Epoch đã chạy  : 194 (early stopping, patience=60)
Thời gian train: NumPy 3.76s | PyTorch 4.15s
Test Accuracy  : 0.7500  (@0.5)
Test F1        : 0.6420  (@0.5)
Test ROC-AUC   : 0.8382
Ngưỡng tối ưu  : 0.430 -> Acc=0.7586, F1=0.6889, Recall=0.7750

